# Phase Loop: Batch Phase Search over Multiple .xy Files
Runs the DARA tree-search pipeline from `phase_search copy.ipynb` over every `.xy` file
found in `all_xy_files/`. CIFs are downloaded once and reused for all patterns.

In [ ]:
# %pip install ipywidgets nbformat

In [ ]:
import os
import re
import time
import tempfile
import requests
from pathlib import Path

from dara import search_phases
from dara.cif import Cif
from dara.structure_db import CODDatabase
from pymatgen.io.cif import CifParser

# ── Configuration ────────────────────────────────────────────────────────────
DATA_DIR       = Path("all_xy_files")   # folder containing .xy files to process
CIFS_DIR       = Path("cifs")           # shared CIF cache (reused across patterns)
CHEMICAL_SYSTEM = "Li-Ni-Mn-Co-O-H"
WAVELENGTH      = "Cu"
INSTRUMENT      = "Aeris-fds-Pixcel1d-Medipix3"
EXPRESS_MODE    = False
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR.mkdir(exist_ok=True)
CIFS_DIR.mkdir(exist_ok=True)
print(f"Looking for .xy files in: {DATA_DIR.resolve()}")

## Step 1: Helpers — header stripper & CIF lattice reader

In [ ]:
def strip_xy_header(path: Path) -> Path:
    """Return path to a header-stripped copy of the .xy file.

    COMMANDER-exported .xy files have metadata lines at the top that
    DARA cannot parse. Strips non-numeric lines and writes a *_clean.xy
    sibling file. Idempotent: skips re-creation if clean file already exists
    and is newer than the source.
    """
    clean_path = path.parent / (path.stem + "_clean.xy")
    if clean_path.exists() and clean_path.stat().st_mtime >= path.stat().st_mtime:
        return clean_path
    lines = path.read_text(encoding="utf-8").splitlines()
    data_lines = [l for l in lines if re.match(r"^\s*[\d.\-]", l)]
    clean_path.write_text("\n".join(data_lines), encoding="utf-8")
    return clean_path


def cif_stem_to_bgmn_key(stem: str) -> str:
    """Map a CIF filename stem to the BGMN phase key.

    E.g. 'LiNiO2_166_(cod_1533770)-13' -> 'LiNiO2166cod153377013'
    """
    return re.sub(r"[_()\-]", "", stem)


def get_literature_params_nm(cif_path: Path):
    """Return (a, b, c) in nm from a CIF file. Returns (None, None, None) on failure."""
    try:
        parser = CifParser(str(cif_path), occupancy_tolerance=10.0)
        structure = parser.parse_structures(primitive=False)[0]
        lat = structure.lattice
        return lat.a / 10, lat.b / 10, lat.c / 10
    except Exception as exc:
        print(f"    [CIF read error] {cif_path.name}: {exc}")
        return None, None, None

## Step 2: Prepare reference CIFs (downloaded once, cached to `cifs/`)

In [ ]:
# # Windows fix: COD download retry with exponential back-off
# _COD_SESSION = requests.Session()
# _COD_SESSION.headers.update({
#     "User-Agent": "Mozilla/5.0 (research use; contact: your@email.com)"
# })
# COD_URL = "https://www.crystallography.net/cod/{cod_id}.cif"

# def _download_cod_windows_fix(cod_id: str) -> Cif:
#     max_retries = 5
#     for attempt in range(max_retries):
#         try:
#             time.sleep(0.5 + attempt * 1.5)
#             url = COD_URL.format(cod_id=cod_id)
#             response = _COD_SESSION.get(url, timeout=30)
#             response.raise_for_status()

#             # Windows fix: NamedTemporaryFile keeps the handle open and blocks
#             # unlink; use mkstemp for explicit fd control instead.
#             fd, temp_path = tempfile.mkstemp(suffix=".cif")
#             try:
#                 os.write(fd, response.content)
#                 os.close(fd)   # release handle BEFORE Cif.from_file opens it
#                 fd = -1        # mark as closed so finally-block skips it
#                 cif = Cif.from_file(temp_path)
#             finally:
#                 if fd != -1:   # close fd if an error occurred before os.close()
#                     try:
#                         os.close(fd)
#                     except OSError:
#                         pass
#                 for _ in range(5):   # retry unlink: Windows may hold the lock briefly
#                     try:
#                         Path(temp_path).unlink(missing_ok=True)
#                         break
#                     except PermissionError:
#                         time.sleep(0.2)

#             cif.filename = str(cod_id)
#             return cif
#         except Exception as e:
#             if attempt < max_retries - 1:
#                 wait = 2 ** attempt
#                 print(f"  [{cod_id}] attempt {attempt+1} failed ({e}), retrying in {wait}s…")
#                 time.sleep(wait)
#             else:
#                 print(f"  [{cod_id}] all {max_retries} attempts failed: {e}")
#                 raise

# CODDatabase._download_cod = staticmethod(_download_cod_windows_fix)

# cod_database = CODDatabase()
# all_icsd_ids = cod_database.get_cifs_by_chemsys(CHEMICAL_SYSTEM, dest_dir=str(CIFS_DIR))
# print(f"CIF cache ready: {len(list(CIFS_DIR.glob('*.cif')))} files in {CIFS_DIR}")

In [ ]:
# Windows fix: BGMN holds a file lock on .dia/.str temp files briefly after
# the subprocess exits. Python's TemporaryDirectory.__exit__ raises
# PermissionError [WinError 32] when it tries to unlink those files.
# Setting ignore_cleanup_errors=True (Python 3.10+) silences the cleanup
# error; the OS reclaims the files on next reboot / temp sweep.
import tempfile as _tempfile

_OrigTempDir = _tempfile.TemporaryDirectory

class _WindowsTempDir(_OrigTempDir):
    def __init__(self, *args, **kwargs):
        kwargs.setdefault("ignore_cleanup_errors", True)
        super().__init__(*args, **kwargs)

_tempfile.TemporaryDirectory = _WindowsTempDir
print("Windows fix applied: TemporaryDirectory cleanup errors will be ignored.")

## Step 3: Initialise Ray and build curated CIF list

In [ ]:
import ray

# Windows fix: stale Ray instances from a previous run cause PermissionError.
if ray.is_initialized():
    ray.shutdown()
ray.init(runtime_env={"working_dir": None})

curated_cifs = [f for f in CIFS_DIR.resolve().glob("*.cif")]
print(f"Curated set: {len(curated_cifs)} phases")

In [ ]:
# Windows fix: patch remote_do_refinement_no_saving so the tempdir cleanup
# fix runs INSIDE each Ray worker process, not just in the notebook kernel.
#
# The PermissionError [WinError 32] on .dia files happens in the worker because
# BGMN briefly holds a file lock after its subprocess exits. The kernel-side
# _WindowsTempDir patch never reaches workers (separate OS processes). This
# wrapper redefines the remote function so the fix is applied in the worker
# before do_refinement_no_saving opens its TemporaryDirectory.

import ray
import dara.search.tree as _dara_tree
from dara.refine import do_refinement_no_saving as _do_refinement_no_saving

@ray.remote(num_cpus=1)
def _patched_remote_refinement(
    pattern_path,
    cif_paths,
    wavelength,
    instrument_profile,
    phase_params,
    refinement_params,
):
    import tempfile, shutil, time
    import dara.refine as _dr

    # Apply the fix locally inside this worker process
    _OrigTD = tempfile.TemporaryDirectory

    class _RobustTempDir(_OrigTD):
        def __init__(self, *args, **kwargs):
            kwargs.setdefault("ignore_cleanup_errors", True)
            super().__init__(*args, **kwargs)

        def cleanup(self):
            for attempt in range(6):
                try:
                    shutil.rmtree(self.name)
                    return
                except PermissionError:
                    if attempt < 5:
                        time.sleep(0.2 * (attempt + 1))
            shutil.rmtree(self.name, ignore_errors=True)

    tempfile.TemporaryDirectory = _RobustTempDir
    _dr.tempfile.TemporaryDirectory = _RobustTempDir

    # Replicate the original remote function's logic exactly
    from subprocess import TimeoutExpired
    from dara.cif2str import CIF2StrError
    import logging
    logger = logging.getLogger("dara.search.tree")

    if len(cif_paths) == 0:
        return None
    try:
        result = _do_refinement_no_saving(
            pattern_path,
            cif_paths,
            wavelength=wavelength,
            instrument_profile=instrument_profile,
            phase_params=phase_params,
            refinement_params=refinement_params,
        )
    except (RuntimeError, TimeoutExpired, CIF2StrError, ValueError) as e:
        logger.debug(f"Refinement failed for {cif_paths}, the reason is {e}")
        return None
    if result.lst_data.rpb == 100:
        logger.debug(f"Refinement failed for {cif_paths}, the reason is RPB = 100.")
        return None
    return result

# Replace DARA's remote function so batch_refinement picks up our wrapper
_dara_tree.remote_do_refinement_no_saving = _patched_remote_refinement
print("Worker-side Windows fix applied: remote_do_refinement_no_saving patched.")

## Step 4: Batch loop — run phase search on every .xy file

In [ ]:
xy_files = sorted(DATA_DIR.glob("*.xy"))
# Exclude already-cleaned copies so we don't process them twice
xy_files = [f for f in xy_files if not f.stem.endswith("_clean")]

if not xy_files:
    print(f"No .xy files found in {DATA_DIR.resolve()}. Add your files and re-run.")
else:
    print(f"Found {len(xy_files)} .xy file(s):")
    for f in xy_files:
        print(f"  {f.name}")

In [ ]:
all_results = {}   # pattern_name -> list[SearchResult]

for xy_file in xy_files:
    print(f"\n{'='*70}")
    print(f"Processing: {xy_file.name}")
    print(f"{'='*70}")

    # Strip header (idempotent)
    clean_path = strip_xy_header(xy_file)
    print(f"  Clean file: {clean_path.name}")

    # Ray must be alive; re-init if it died between iterations
    if not ray.is_initialized():
        ray.init(runtime_env={"working_dir": None})

    search_results = search_phases(
        pattern_path=str(clean_path),
        phases=curated_cifs,
        wavelength=WAVELENGTH,
        instrument_profile=INSTRUMENT,
        express_mode=EXPRESS_MODE,
        enable_angular_cut=False,
        phase_params = {'lattice_range' : 0.03, 
                        "b1": "0.003_0.0^0.01"},
        rpb_threshold = 0.5
    )

    all_results[xy_file.name] = search_results

    # Quick summary for this file
    for i, sr in enumerate(search_results):
        rwp = sr.refinement_result.lst_data.rwp
        top_phase = list(sr.refinement_result.lst_data.phases_results.keys())[0]
        print(f"  Solution {i}: Rwp={rwp:.2f}%  top phase={top_phase}")
        sr.visualize()

print(f"\nDone. Processed {len(all_results)} file(s).")

In [ ]:
sr.visualize()

In [ ]:
print("Missing peaks (2θ, intensity):", sr.missing_peaks)
print("Extra peaks   (2θ, intensity):", sr.extra_peaks)

## Step 5: Summary table — best solution per file

In [ ]:
import pandas as pd

rows = []
for fname, results in all_results.items():
    if not results:
        rows.append({"file": fname, "rwp": None, "top_phase": "no result",
                     "a_nm": None, "c_nm": None})
        continue
    best = results[0]  # DARA returns solutions sorted by FOM; take first
    rwp  = best.refinement_result.lst_data.rwp
    phases_results = best.refinement_result.lst_data.phases_results
    top_key = list(phases_results.keys())[0]
    pr  = phases_results[top_key]
    a_nm = pr.a[0] if pr.a else None
    c_nm = pr.c[0] if pr.c else None
    rows.append({
        "file":      fname,
        "rwp":       rwp,
        "top_phase": top_key,
        "a_nm":      round(a_nm, 6) if a_nm else None,
        "c_nm":      round(c_nm, 6) if c_nm else None,
    })

df_summary = pd.DataFrame(rows)
display(df_summary)

## Step 6: Detailed lattice parameters — refined vs literature

In [ ]:
for fname, results in all_results.items():
    print(f"\n{'='*70}")
    print(f"File: {fname}")
    print(f"{'='*70}")

    for sol_idx, result in enumerate(results):
        rwp = result.refinement_result.lst_data.rwp
        print(f"\n  Solution {sol_idx}  (Rwp = {rwp:.2f} %)")
        print(f"  {'-'*66}")

        phases_results = result.refinement_result.lst_data.phases_results
        best_phases    = result.phases[0]
        key_to_path    = {cif_stem_to_bgmn_key(rp.path.stem): rp.path for rp in best_phases}

        for phase_key, pr in phases_results.items():
            cif_path = key_to_path.get(phase_key)
            lit_a, lit_b, lit_c = get_literature_params_nm(cif_path) if cif_path else (None, None, None)
            label = cif_path.name if cif_path else phase_key

            print(f"\n    Phase: {label}")
            print(f"    {'Param':<8} {'Refined (nm)':>14} {'± σ':>10} {'Literature (nm)':>16} {'Δ (%)':>8}")
            print(f"    {'-'*58}")

            for pname, refined_tuple, lit_val in [("a", pr.a, lit_a), ("b", pr.b, lit_b), ("c", pr.c, lit_c)]:
                if refined_tuple is None:
                    continue
                ref_val, ref_err = refined_tuple
                lit_str   = f"{lit_val:.6f}" if lit_val is not None else "N/A"
                err_str   = f"± {ref_err:.6f}" if ref_err is not None else ""
                delta_str = f"{100 * (ref_val - lit_val) / lit_val:+.3f}%" if lit_val else ""
                print(f"    {pname:<8} {ref_val:>14.6f} {err_str:>10} {lit_str:>16} {delta_str:>8}")

In [ ]:
# ======================================================================
# File: OLM-CM 0506026_exported.xy
# ======================================================================

#   Solution 0  (Rwp = 10.85 %)
#   ------------------------------------------------------------------

#     Phase: LiCoO2_166_(cod_4505482)-0
#     Param      Refined (nm)        ± σ  Literature (nm)    Δ (%)
#     ----------------------------------------------------------
#     a              0.286539 ± 0.000006              N/A         
#     c              1.424397 ± 0.000055              N/A         

#     Phase: Li2.772Ni3.228O6_166_(cod_1525553)-None
#     Param      Refined (nm)        ± σ  Literature (nm)    Δ (%)
#     ----------------------------------------------------------
#     a              0.286920 ± 0.000025              N/A         
#     c              1.425450 ± 0.000220              N/A         

#   Solution 1  (Rwp = 11.58 %)
#   ------------------------------------------------------------------

#     Phase: Li0.65Ni1.08O2_166_(cod_1535512)-None
#     Param      Refined (nm)        ± σ  Literature (nm)    Δ (%)
#     ----------------------------------------------------------
#     a              0.286614 ± 0.000006              N/A         
#     c              1.424585 ± 0.000057              N/A  

In [ ]:

# ======================================================================
# File: NM ANODE CAHODE (NM-CI) 2305026_exported.xy
# ======================================================================

#   Solution 0  (Rwp = 10.09 %)
#   ------------------------------------------------------------------

#     Phase: Mn1.92O4_225_(cod_4001899)-None
#     Param      Refined (nm)        ± σ  Literature (nm)    Δ (%)
#     ----------------------------------------------------------
#     a              0.408577 ± 0.000029              N/A         

#     Phase: Li1.26Ni2.04O4_166_(cod_1526282)-None
#     Param      Refined (nm)        ± σ  Literature (nm)    Δ (%)
#     ----------------------------------------------------------
#     a              0.498538 ± 0.000019              N/A         
#     b              0.287331 ± 0.000011              N/A         
#     c              0.502202 ± 0.000022              N/A         

#     Phase: Li8.24Mn15.76O32_227_(cod_1514043)-None
#     Param      Refined (nm)        ± σ  Literature (nm)    Δ (%)
#     ----------------------------------------------------------
#     a              0.813090 ± 0.000150              N/A         

#   Solution 1  (Rwp = 11.50 %)
#   ------------------------------------------------------------------

#     Phase: LiNiO2_166_(cod_1533770)-13
#     Param      Refined (nm)        ± σ  Literature (nm)    Δ (%)
#     ----------------------------------------------------------
#     a              0.287631 ± 0.000008              N/A         
#     c              1.420708 ± 0.000070              N/A         